In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

# --- Load dataset ---
df = pd.read_csv("/content/DM1_game_dataset_module.01.csv")

# Convert Rating to numeric
rating_map = {"Low": 0, "Medium": 1, "High": 2}
df["Rating"] = df["Rating"].map(rating_map)

# Feature setups
features_uni = ["NumWish"]
features_multi2 = ["NumWish", "NumUserRatings"]
features_multi3 = ["NumWish", "NumUserRatings", "NumOwned"]

# All numeric features except Rating
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols.remove("Rating")  # remove target
features_all = numeric_cols

feature_sets = {
    "Univariate": features_uni,
    "Multivariate (2)": features_multi2,
    "Multivariate (3)": features_multi3,
    "All Variables": features_all
}

models = {
    "Linear": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(max_iter=5000),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "KNN": KNeighborsRegressor(n_neighbors=5)
}

results = {m: {} for m in models.keys()}

for config_name, feats in feature_sets.items():
    X = df[feats]
    y = df["Rating"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for model_name, model in models.items():
        if model_name == "KNN":  # Only KNN uses scaling
            model.fit(X_train_scaled, y_train)
            y_pred = model.predict(X_test_scaled)
        else:
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)

        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        mse = mean_squared_error(y_test, y_pred)

        results[model_name][config_name] = f"{r2:.3f}, {mae:.3f}, {mse:.3f}"

# Final comparison table
final_table = pd.DataFrame(results).T[
    ["All Variables", "Univariate", "Multivariate (2)", "Multivariate (3)"]
]

final_table


,All Variables,Univariate,Multivariate (2),Multivariate (3)
Linear,"0.557, 0.397, 0.246","0.091, 0.579, 0.505","0.101, 0.575, 0.499","0.102, 0.576, 0.499"
Ridge,"0.541, 0.404, 0.255","0.091, 0.579, 0.505","0.101, 0.575, 0.499","0.102, 0.576, 0.499"
Lasso,"0.421, 0.454, 0.321","0.091, 0.579, 0.505","0.101, 0.575, 0.500","0.103, 0.576, 0.498"
Decision Tree,"0.351, 0.330, 0.360","0.281, 0.506, 0.399","0.016, 0.511, 0.547","-0.104, 0.508, 0.613"
KNN,"0.435, 0.427, 0.314","0.182, 0.544, 0.454","0.323, 0.474, 0.376","0.334, 0.465, 0.370"
